In [1]:
import pyspark

In [2]:
from pyspark.sql import SparkSession
# spark 2.0 이후, sparkContext와 동일한 기능 (spark cluster의 entry point)
# sparkContext도 여전히 지원하며, RDD를 이용할 경우, sparkContext를 사용함

## `SparkSession` 생성

- `SparkSession.builder`를 사용하여 기본 설정의 `SparkSession` 생성
- `config('spark.driver.host','localhost')`: application의 host를 local로 설정(환경변수 설정 필요)
- `appName`: application 이름 설정
- `getOrCreate`: 기존 `SparkSession`을 가져오거나, 새롭게 생성함

In [3]:
spark = SparkSession.builder.appName('01').config('spark.driver.host','localhost').getOrCreate()

## `Data read`

- `SparkSession`의 `read`는 데이터를 로드한 후, `DataFrame` type으로 반환함
- 일반적으로 local에서 데이터를 로드하는 경우, `file:///` 또는 `C:///`이후의 full path를 입력
- cluster 환경에서는 **url** 또는 **HDFS**일 수 있다.

In [6]:
students = spark.read.csv('C:\\Users\\User\\Downloads\\3rd_week_data\\students.csv')

- 우리는 데이터가 cluster환경에서 분산되어 저장된 큰 용량의 데이터로 가정함
- 그러므로 지금 현재 driver가 실행 중인 host에 모든 데이터를 가져와서 확인하는 것은 불가능하다고 간주함
- `print`를 이용한 `DataFrame`전체 출력은 불가하며, `show`를 통해 일부만 확인할 수 있음
- `DataFrame.show(n)`: n개의 행만 반환

In [9]:
students.show(5)

+------+--------------+--------------------+------------+--------------------+----------+-------------+-------------+
|   _c0|           _c1|                 _c2|         _c3|                 _c4|       _c5|          _c6|          _c7|
+------+--------------+--------------------+------------+--------------------+----------+-------------+-------------+
|gender|race/ethnicity|parental level of...|       lunch|test preparation ...|math score|reading score|writing score|
|female|       group B|   bachelor's degree|    standard|                none|        72|           72|           74|
|female|       group C|        some college|    standard|           completed|        69|           90|           88|
|female|       group B|     master's degree|    standard|                none|        90|           95|           93|
|  male|       group A|  associate's degree|free/reduced|                none|        47|           57|           44|
+------+--------------+--------------------+------------

- `DataFrame.printSchema()`: DataFrame의 **Schema** 확인
> - column명, data type, nullable 정보 

In [10]:
students.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)



- `header`: True로 설정할 경우, 첫행을 column명으로 간주함 (`default: None`)
- `inferSchema`
> - `True`로 설정할 경우, 각 **column의 type과 nullable**를 확인
> - `default = None`: 모든 column의 type을 **string으로 간주**
> - 대용량 데이터의 경우, True로 설정하면 read 속도가 매우 느릴 수 있으므로, **직접 `schema`를 입력**하는 것이 좋음

In [14]:
students = spark.read.csv('C:\\Users\\User\\Downloads\\3rd_week_data\\students.csv',
                          header=True,
                          inferSchema=True)
students.printSchema()

root
 |-- gender: string (nullable = true)
 |-- race/ethnicity: string (nullable = true)
 |-- parental level of education: string (nullable = true)
 |-- lunch: string (nullable = true)
 |-- test preparation course: string (nullable = true)
 |-- math score: integer (nullable = true)
 |-- reading score: integer (nullable = true)
 |-- writing score: integer (nullable = true)



- `json` 파일 로드
> - json type의 경우, header 설정하지 않아도 column명이 자동으로 할당됨 

In [15]:
people = spark.read.json('C:\\Users\\User\\Downloads\\3rd_week_data\\people.json')
print(type(people))
people.show(3)
people.printSchema()

<class 'pyspark.sql.dataframe.DataFrame'>
+---------------+---------------+-------------------+--------------------+-----------------+------------+--------------------+
|_corrupt_record|           city|         creditcard|               email|              mac|        name|           timestamp|
+---------------+---------------+-------------------+--------------------+-----------------+------------+--------------------+
|              [|           NULL|               NULL|                NULL|             NULL|        NULL|                NULL|
|           NULL|Lake Gladysberg|1228-1221-1221-1431|katlyn@jenkinsmag...|08:fd:0b:cd:77:f7|Keeley Bosco|2015-04-25 13:57:...|
|           NULL|           NULL|1228-1221-1221-1431|juvenal@johnston....|90:4d:fa:42:63:a2| Rubye Jerde|2015-04-25 09:02:...|
+---------------+---------------+-------------------+--------------------+-----------------+------------+--------------------+
only showing top 3 rows

root
 |-- _corrupt_record: string (nullable 

In [18]:
print(people) # 각 column의 이름과 타입 확인
print(people.columns) # 각 column의 이름 확인

DataFrame[_corrupt_record: string, city: string, creditcard: string, email: string, mac: string, name: string, timestamp: string]
['_corrupt_record', 'city', 'creditcard', 'email', 'mac', 'name', 'timestamp']


In [19]:
print(students.describe())
# column명과 타입 반환, 
# printSchema 정보와 중복되므로, 주로 printSchema 사용

DataFrame[summary: string, gender: string, race/ethnicity: string, parental level of education: string, lunch: string, test preparation course: string, math score: string, reading score: string, writing score: string]


## `Schema` 지정하기

- read할 데이터의 `Schema`를 미리 지정해서, `inferSchema`를 `True`로 설정하는 것보다 빠른 속도로 로드할 수 있도록 함

In [20]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, DateType

- `StructField`: 한 개의 column을 정의함 (`name`, `data type`, `nullable`)
- `StructType`: 여러 개의 `StructField`가 모인 list형태의 container(`Schema`)

In [21]:
# time은 DateType, 나머지 column은 StringType으로 지정함
people_field = [
    StructField('name',StringType(),True),
    StructField('email',StringType(),True),
    StructField('city',StringType(),True),
    StructField('mac',StringType(),True),
    StructField('time',DateType(),True),
    StructField('creditcard',StringType(),True)
]

In [22]:
# StructField를 원소로하는 list로 StructType 생성
people_schema = StructType(fields=people_field)

In [23]:
people2 = spark.read.json('C:\\Users\\User\\Downloads\\3rd_week_data\\people.json',schema=people_schema)

In [24]:
people.printSchema()
people2.printSchema()

root
 |-- _corrupt_record: string (nullable = true)
 |-- city: string (nullable = true)
 |-- creditcard: string (nullable = true)
 |-- email: string (nullable = true)
 |-- mac: string (nullable = true)
 |-- name: string (nullable = true)
 |-- timestamp: string (nullable = true)

root
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- mac: string (nullable = true)
 |-- time: date (nullable = true)
 |-- creditcard: string (nullable = true)

